<a href="https://colab.research.google.com/github/ankurdev1-drth/Deep_Learning-/blob/main/Building_Neural_Network_Using_PyTorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## PyTorch Fundamentals

Note:
- core data structure Pytorch  tensor
- tensor:
   - multidimensional array with a shape and a data type, used for numerical computations
   - it can live on a GPU
   - it supports auto-differentiation
   - a tensor can contain floats, integers, booleans, or complex numbers—just one data type per tensor.
   - Indexing works just like for NumPy arrays

### PyTorch Tensors

In [10]:
import torch

In [11]:
X = torch.tensor([[1.0, 4.0, 7.0], [2.0, 3.0, 6.0]])

In [12]:
X

tensor([[1., 4., 7.],
        [2., 3., 6.]])

In [13]:
X.shape

torch.Size([2, 3])

In [14]:
X.dtype

torch.float32

In [15]:
X[0,1]

tensor(4.)

In [16]:
X[1,1]

tensor(3.)

In [17]:
X[:, 1]  #shows everything from column 1

tensor([4., 3.])

In [18]:
10 *(X + 1.0) # itemise addition and multiplication

tensor([[20., 50., 80.],
        [30., 40., 70.]])

In [19]:
X.exp() #itemwise exponential

tensor([[   2.7183,   54.5981, 1096.6332],
        [   7.3891,   20.0855,  403.4288]])

In [20]:
X.mean()

tensor(3.8333)

In [21]:
X.max(dim=0) #max values along dimension 0 (max value per column)

torch.return_types.max(
values=tensor([2., 4., 7.]),
indices=tensor([1, 0, 0]))

In [22]:
X @ X.T #matrix transpose and matrix multiplication

tensor([[66., 56.],
        [56., 49.]])

Note:
- PyTorch prefers the argument name dim in operations such as max() , but it also supports axis (as in NumPy or Pandas).
- You can also convert a tensor to a NumPy array using the numpy() method, and create a tensor from a NumPy array
- torch.FloatTensor()  automatically converts the array to 32 bits
- modification of a tensor in place using indexing and slicing is possible
- PyTorch’s API provides many in-place operations, such as abs_() , sqrt_() , and zero_() , which modify the input tensor directly
-

In [23]:
import numpy as np
X.numpy()

array([[1., 4., 7.],
       [2., 3., 6.]], dtype=float32)

In [24]:
torch.tensor(np.array([[1., 4., 7.], [2., 3., 6.]]), dtype=torch.float32)

tensor([[1., 4., 7.],
        [2., 3., 6.]])

In [25]:
X[:, 1] =  -99

In [26]:
X

tensor([[  1., -99.,   7.],
        [  2., -99.,   6.]])

In [27]:
X[1,  :] = 0

In [28]:
X

tensor([[  1., -99.,   7.],
        [  0.,   0.,   0.]])

In [29]:
X.relu_()  #here relu_() replaces all the negative numbers with 0

tensor([[1., 0., 7.],
        [0., 0., 0.]])

## Hardware Acceleration

In [30]:
if torch.cuda.is_available():
    device="cuda"
elif torch.backends.mps.is_available():
    device="mps"
else:
    device="cpu"

In [31]:
M = torch.tensor([[1., 2., 3.], [4., 5., 6.]])

In [32]:
M = M.to(device)

In [33]:
M.device #tells on which device tensor lives on

device(type='cuda', index=0)

We can create tensor on gpu using *device* argument

In [34]:
M = torch.tensor([[1., 2., 3.], [4., 5., 6.]], device=device)

In [35]:
M

tensor([[1., 2., 3.],
        [4., 5., 6.]], device='cuda:0')

Now all the operations will run on gpu

In [36]:
R = M @ M.T

In [37]:
R

tensor([[14., 32.],
        [32., 77.]], device='cuda:0')

The result R also lives on GPU instead of CPU that means we can perform multiple operations on GPU

In [38]:
#testing the speed of the multiplication matrix on
M = torch.rand((1000,1000)) # on cpu

In [39]:
%timeit M @ M.T

15.9 ms ± 1.09 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [40]:
M = torch.rand((1000, 1000), device="cuda")

In [41]:
%timeit M@M.T

519 µs ± 5.91 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)


## Autograd (automatic gradient)

Note:
- It's an implementation of reverse-mode, auto-differentiation

In [42]:
#taking an example:
x = torch.tensor(5.0, requires_grad=True)
f = x**2
f

tensor(25., grad_fn=<PowBackward0>)

In [43]:
f.backward()

In [44]:
x.grad

tensor(10.)

backward()
function automatically computed the gradient f'(x) at the same
point x = 5.0.

Note:
- we told PyTorch that it’s a variable (not a constant) by specifying requires_grad=True
- tensor x is a leaf node
- f also carries a grad_fn attribute which represents the operation that created this tensor ( ** , power, hence the name PowBackward0),and which tells PyTorch how to backpropagate the gradients through this particular operation.
- we call f.backward() : this backpropagates the gradients through the computation graph, starting with f , and all the way back to the leaf nodes (just x in this case)


In [45]:
#performing gradient descent
learning_rate = 0.1
with torch.no_grad():
    x -= learning_rate * x.grad ##gradient descent step


Another way to avoid gradient computation is to use the variable’s detach()
method: this creates a new tensor detached from the computation graph, with
requires_grad=False , but still pointing to the same data in memory.

In [46]:
x_detached = x.detach()

In [47]:
x_detached -= learning_rate * x.grad

In [48]:
x_detached

tensor(3.)

In [49]:
X.tanh_()

tensor([[0.7616, 0.0000, 1.0000],
        [0.0000, 0.0000, 0.0000]])

In [50]:
X.sigmoid_()

tensor([[0.6817, 0.5000, 0.7311],
        [0.5000, 0.5000, 0.5000]])

In [51]:
x.grad.zero_()

tensor(0.)

In [52]:
learning_rate = 0.1
x = torch.tensor(5.0, requires_grad=True)
for iteration in range(100):
    f = x**2
    f.backward()
    with torch.no_grad():
        x -=learning_rate * x.grad # gradient descent step

    x.grad.zero_() #reset the gradients

In [53]:
t = torch.tensor(2, requires_grad = True, dtype = float)
z = t.cos()+1 # intermediate result
z += 1 # in-place operation
z.backward()


Note:
- operations—such as exp() , relu() , rsqrt() , sigmoid() , sqrt() ,tan() , and tanh() —save their outputs in the computation graph duringthe forward pass, then use these outputs to compute the gradients durin the backward pass.5 This means that you must not modify such an operation’s output in place, or you will get an error during the backward pass
- operations—such as abs() , cos() , log() , sin() , square() , and var() —save their inputs instead of their output.6 Such an operation
doesn’t care if you modify its output in place, but you must not modify its inputs in place before the backward pass
- rations—such as max() , min() , norm() , prod() , sgn() , and std() —save both the inputs and the outputs, so you must not modify
either of them in place before the backward pass.


## Implementing Linear Regression

### Linear Regression Using Tensors and Autograd

In [54]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

housing = fetch_california_housing()
X_trainf, X_test, y_trainf, y_test = train_test_split(housing.data, housing.target, random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(X_trainf, y_trainf, random_state=42)

In [55]:
housing.data

array([[   8.3252    ,   41.        ,    6.98412698, ...,    2.55555556,
          37.88      , -122.23      ],
       [   8.3014    ,   21.        ,    6.23813708, ...,    2.10984183,
          37.86      , -122.22      ],
       [   7.2574    ,   52.        ,    8.28813559, ...,    2.80225989,
          37.85      , -122.24      ],
       ...,
       [   1.7       ,   17.        ,    5.20554273, ...,    2.3256351 ,
          39.43      , -121.22      ],
       [   1.8672    ,   18.        ,    5.32951289, ...,    2.12320917,
          39.43      , -121.32      ],
       [   2.3886    ,   16.        ,    5.25471698, ...,    2.61698113,
          39.37      , -121.24      ]])

In [56]:
housing.target

array([4.526, 3.585, 3.521, ..., 0.923, 0.847, 0.894])

In [57]:
X_train = torch.FloatTensor(X_train)
X_test = torch.FloatTensor(X_test)
X_valid = torch.FloatTensor(X_valid)
means = X_train.mean(dim=0, keepdims=True)
stds = X_train.std(dim=0, keepdims=True)
X_train = (X_train- means) / stds
X_valid = (X_valid - means) / stds
X_test = (X_test - means) / stds


In [58]:
# as the target is a 1D array we need to reshape he tensors to column vectors by adding  second dimension f size 1 :
y_train = torch.FloatTensor(y_train).reshape(-1,1)
y_test = torch.FloatTensor(y_test).reshape(-1,1)
y_valid = torch.FloatTensor(y_valid).reshape(-1,1)

In [59]:
y_train

tensor([[1.4420],
        [1.6870],
        [1.6210],
        ...,
        [0.6800],
        [0.6130],
        [1.9700]])

In [60]:
## creating the parameters of our linear regression

torch.manual_seed(42)
n_features = X_train.shape[1] # there are 8 input features
w = torch.randn((n_features, 1), requires_grad = True)
b = torch.tensor(0., requires_grad=True)

In [61]:
#training model with help of BGD(batch-gradient descent) using full training set at each step
learning_rate=0.4
n_epochs=20
for epoch in range(n_epochs):
    y_pred = X_train @ w + b   #running the forward pass
    loss = ((y_pred - y_train) **2).mean()
    loss.backward() # computing the gradients of the loss with regard to every model parameter. this is an autograd in action
    with torch.no_grad():
        b -= learning_rate * b.grad # performing gradient descent
        w -= learning_rate * w.grad # performing gradient descent
        b.grad.zero_() # reducing the gradients to zero
        w.grad.zero_()
    print(f"Epoch {epoch + 1}/ {n_epochs}, Loss: {loss.item()}") # .item method extracts the value of the scalar tensor

Epoch 1/ 20, Loss: 16.158456802368164
Epoch 2/ 20, Loss: 4.8793745040893555
Epoch 3/ 20, Loss: 2.255225419998169
Epoch 4/ 20, Loss: 1.3307634592056274
Epoch 5/ 20, Loss: 0.9680691957473755
Epoch 6/ 20, Loss: 0.8142675757408142
Epoch 7/ 20, Loss: 0.7417045831680298
Epoch 8/ 20, Loss: 0.7020701169967651
Epoch 9/ 20, Loss: 0.6765918731689453
Epoch 10/ 20, Loss: 0.6577965021133423
Epoch 11/ 20, Loss: 0.6426151990890503
Epoch 12/ 20, Loss: 0.6297222971916199
Epoch 13/ 20, Loss: 0.6184942126274109
Epoch 14/ 20, Loss: 0.6085968613624573
Epoch 15/ 20, Loss: 0.5998216867446899
Epoch 16/ 20, Loss: 0.592018723487854
Epoch 17/ 20, Loss: 0.5850691795349121
Epoch 18/ 20, Loss: 0.578873336315155
Epoch 19/ 20, Loss: 0.573345422744751
Epoch 20/ 20, Loss: 0.5684100389480591


In [62]:
# making predictions for the first three instances in the test set

X_new = X_test[:3] #pretending that these are new instances
with torch.no_grad():
    y_pred = X_new @ w + b # using the trained parameters to make predictions
y_pred


tensor([[0.8916],
        [1.6480],
        [2.6577]])

**with torch.no_grad()** PyTorch takes less ram as it doesn't need to keep track of the computation graph

## Linear Regression Using PyTorch's High-Level API

**torch.nn.Linear** class in PyTorch is used for implementation of Linear Regression

In [63]:
import torch.nn as nn

torch.manual_seed(42) #for getting reproducible results
model = nn.Linear(in_features=n_features, out_features=1)

In [64]:
model.bias

Parameter containing:
tensor([0.3117], requires_grad=True)

In [65]:
model.weight

Parameter containing:
tensor([[ 0.2703,  0.2935, -0.0828,  0.3248, -0.0775,  0.0713, -0.1721,  0.2076]],
       requires_grad=True)

In [66]:
for param in model.parameters():
  print(param)

Parameter containing:
tensor([[ 0.2703,  0.2935, -0.0828,  0.3248, -0.0775,  0.0713, -0.1721,  0.2076]],
       requires_grad=True)
Parameter containing:
tensor([0.3117], requires_grad=True)


In [67]:
model(X_train[:2])

tensor([[-0.4718],
        [ 0.1131]], grad_fn=<AddmmBackward0>)

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
mse = nn.MSELoss()

In [ ]:
# writing a function to train the model
def train_bgd(model, optimizer, criterion, X_train, y_train, n_epochs):
  for epoch in range(n_epochs):
    y_pred = model(X_train)
    loss = criterion(y_pred, y_train)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    print(f"Epoch {epoch +1}/ {n_epochs}, Loss:{loss.item()}")

In [ ]:
#calling the function to train the model
train_bgd(model, optimizer, mse, X_train, y_train, n_epochs)


In [ ]:
#model training done! now making predictions by simply calling it like a function
X_new = X_test[:3] #pretend these are new instances
with torch.no_grad():
  y_pred = model(X_new) #use the trained model to make predictions
y_pred

## Implementing a Regression MLP

In [ ]:
torch.manual_seed(42)
model = nn.Sequential(
    nn.Linear(n_features, 50),
    nn.ReLU(),
    nn.Linear(50, 40)
    nn.ReLU(),
    nn.Linear(40, 1)
)

In [ ]:
learning ra